In [1]:
from pathlib import Path
from stable_platform_matchings import InstanceGenerator, Optimizer, OptimizerParams, SolverOptions

In [2]:
SIM_SIZE = 12
N_INTS = 12
SEED = 67

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../../SyntheticInstanceGenerator/data/")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [3]:
ig = InstanceGenerator(
    FARMERS_PATH, FARMERS_14_PATH, INTS_PATH, GRAPH_PATH, ALPHA_PATH, SIGMAS_PATH
)

In [10]:
ig.gen_intermediaries(n_intermediaries=12, seed=12)
ig.gen_calendar(seed=2, scale=2, n_cycles=10)

In [11]:
ig.calendar_df

,scaled_quantity,day,farmer_lat,farmer_lon,intermediary_id,farmer_id
1,0.9,1,-0.326732,102.521687,goofy_kalam,goofy_kalam_d0_f0
2,0.9,16,-0.326732,102.521687,goofy_kalam,goofy_kalam_d0_f0
3,0.9,28,-0.326732,102.521687,goofy_kalam,goofy_kalam_d0_f0
4,0.9,42,-0.326732,102.521687,goofy_kalam,goofy_kalam_d0_f0
5,0.9,60,-0.326732,102.521687,goofy_kalam,goofy_kalam_d0_f0
...,...,...,...,...,...,...
8198,1.4,30,-0.263577,102.319826,wonderful_jepsen,wonderful_jepsen_d13_f1
8199,1.4,41,-0.263577,102.319826,wonderful_jepsen,wonderful_jepsen_d13_f1
8202,1.4,83,-0.263577,102.319826,wonderful_jepsen,wonderful_jepsen_d13_f1
8203,1.4,97,-0.263577,102.319826,wonderful_jepsen,wonderful_jepsen_d13_f1


In [12]:
platform = ig.gen_instance(instance_id="hello", day=69, n_hist_sets=3)

In [13]:
epsilons = {intermediary.id: 2 for intermediary in platform.intermediaries}
het_costs = {
    intermediary.id: (platform.dist_to_mill[intermediary.id] * 2)
    for intermediary in platform.intermediaries
}

In [14]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilons,
    backend="gurobi",
    vrp_mode="approximate",
    verbose=True,
    print_width=80,
)

opt = Optimizer(platform, params)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'affectionate_grothendieck': 437854.80358569464,
   'awesome_hopper': 468775.8359618594,
   'eloquent_hertz': 183689.50503136462,
   'frosty_cerf': 181612.06335893046,
   'goofy_kalam': 459120.9756451647,
   'infallible_morse': 333035.23378009815,
   'optimistic_lalande': 298210.42776725214,
   'relaxed_turing': 515295.89230988536,
   'silly_jemison': 45423.972078040664,
   'stoic_pasteur': 196277.2769834849,
   'stoic_ramanujan': 209572.45446809338,
   'wonderful_jepsen': 756978.9516683007}
----------------------------------- epsilons -----------------------------------
  {'affectionate_grothendieck': 2,
   'awesome_hopper': 2,
   'eloquent_hertz': 2,
   'frosty_cerf': 2,
   'goofy_kalam': 2,
   'infallible_morse': 2,
   'optimistic_lalande': 2,
   'relaxed_turing': 2,
   'silly_jemison': 2,
   'stoic_pasteur': 2,
   'st

In [15]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True,
)


summary = opt.solve(options=options)



================================ Solver Options ================================
  Seed                       0
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False


======================== Strategy: heuristic_optimized =========================
  Farmers                    50
  Intermediaries             12


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

Set parameter Threads to value 1
----------------------------- Primal Solve Result ------------------------------
  New rows                   1027
  Platform profit            887,389.086
  Max intermediary welfare   1,859,001.222
  Max farmer welfare         152,906,618.047
---------------------------- Lower-Bound Candidate -----------------------------
  Objective           

In [17]:
summary.platform_solve_result.platform_profit / platform.lc_to_usd

62.910579289785744